In [1]:
import pandas as pd
import numpy as np
import random

from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

random.seed(42)
np.random.seed(42)

# ======================================
# CONFIG
# ======================================

N_USERS = 5000
N_PRODUCTS = 500

START_DATE = datetime(2025,1,1)
END_DATE = datetime(2025,6,30)

# ======================================
# PRODUCTS
# ======================================

categories = {
    "Electronics": ["Apple","Samsung","Sony","HP"],
    "Fashion": ["Nike","Adidas","Puma","Levis"],
    "Home": ["Ikea","Philips","Prestige"],
    "Sports": ["Yonex","Nivia","Puma"],
    "Books": ["Penguin","Harper","Oxford"],
    "Beauty": ["Loreal","Maybelline","Lakme"]
}

products = []

for pid in range(1, N_PRODUCTS + 1):

    category = random.choice(list(categories.keys()))

    brand = random.choice(categories[category])

    if category == "Electronics":
        price = random.randint(5000, 100000)

    elif category == "Fashion":
        price = random.randint(500, 5000)

    elif category == "Home":
        price = random.randint(1000, 15000)

    elif category == "Sports":
        price = random.randint(300, 8000)

    elif category == "Books":
        price = random.randint(100, 1500)

    else:
        price = random.randint(200, 3000)

    products.append([
        pid,
        category,
        brand,
        price
    ])

products_df = pd.DataFrame(
    products,
    columns=[
        "product_id",
        "category",
        "brand",
        "price"
    ]
)

# ======================================
# USERS
# ======================================

user_types = [
    "Casual",
    "Regular",
    "Power"
]

user_probs = [
    0.60,
    0.30,
    0.10
]

devices = [
    "Android",
    "iOS",
    "Desktop"
]

device_probs = [
    0.55,
    0.25,
    0.20
]

traffic_sources = [
    "Google",
    "Facebook",
    "Organic",
    "Email"
]

traffic_probs = [
    0.40,
    0.20,
    0.25,
    0.15
]

countries = [
    "India",
    "USA",
    "UK",
    "Canada",
    "Australia"
]

users = []

for uid in range(1, N_USERS + 1):

    signup_date = fake.date_between(
        start_date="-365d",
        end_date="today"
    )

    user_type = np.random.choice(
        user_types,
        p=user_probs
    )

    device = np.random.choice(
        devices,
        p=device_probs
    )

    source = np.random.choice(
        traffic_sources,
        p=traffic_probs
    )

    country = random.choice(countries)

    users.append([
        uid,
        signup_date,
        user_type,
        device,
        source,
        country
    ])

users_df = pd.DataFrame(
    users,
    columns=[
        "user_id",
        "signup_date",
        "user_type",
        "device",
        "traffic_source",
        "country"
    ]
)

print("Users:", len(users_df))
print("Products:", len(products_df))

users_df.head()

C:\Users\HP\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Users: 5000
Products: 500


,user_id,signup_date,user_type,device,traffic_source,country
0,1,2026-03-28,Casual,Desktop,Organic,Australia
1,2,2026-02-09,Casual,Android,Google,USA
2,3,2025-11-08,Casual,Desktop,Organic,Canada
3,4,2026-04-13,Regular,Android,Email,Canada
4,5,2026-03-23,Regular,Android,Google,USA


In [2]:
# ======================================
# SESSION GENERATION
# ======================================

sessions = []

session_id = 1

for _, user in users_df.iterrows():

    user_id = user["user_id"]
    user_type = user["user_type"]

    # retention-style behavior

    if user_type == "Casual":
        n_sessions = np.random.randint(2, 8)

    elif user_type == "Regular":
        n_sessions = np.random.randint(8, 20)

    else:
        n_sessions = np.random.randint(20, 40)

    for _ in range(n_sessions):

        # sale boost period
        if random.random() < 0.15:

            session_start = datetime(
                2025,
                3,
                random.randint(15,20),
                random.randint(0,23),
                random.randint(0,59),
                random.randint(0,59)
            )

        else:

            session_start = START_DATE + timedelta(
                seconds=random.randint(
                    0,
                    int(
                        (END_DATE - START_DATE)
                        .total_seconds()
                    )
                )
            )

        variant = np.random.choice(
            ["Control","New_UI"]
        )

        sessions.append([
            session_id,
            user_id,
            session_start,
            variant
        ])

        session_id += 1

sessions_df = pd.DataFrame(
    sessions,
    columns=[
        "session_id",
        "user_id",
        "session_start",
        "experiment_variant"
    ]
)

print("Sessions:", len(sessions_df))

sessions_df.head()

Sessions: 48545


,session_id,user_id,session_start,experiment_variant
0,1,1,2025-02-28 18:21:07,New_UI
1,2,1,2025-02-28 16:54:19,Control
2,3,1,2025-03-14 15:58:17,New_UI
3,4,1,2025-06-01 00:45:57,Control
4,5,1,2025-01-18 23:33:59,New_UI


In [3]:
events = []
orders = []

event_id = 1
order_id = 1

traffic_multiplier = {
    "Google": 1.00,
    "Facebook": 0.75,
    "Organic": 1.30,
    "Email": 1.15
}

device_multiplier = {
    "Android": 0.95,
    "iOS": 1.05,
    "Desktop": 1.10
}

for _, session in sessions_df.iterrows():

    session_id = session["session_id"]
    user_id = session["user_id"]
    variant = session["experiment_variant"]

    user = users_df.loc[
        users_df["user_id"] == user_id
    ].iloc[0]

    source = user["traffic_source"]
    device = user["device"]

    current_time = session["session_start"]

    product_id = random.randint(
        1,
        N_PRODUCTS
    )

    # always starts

    path = ["app_open"]

    # Home View
    if random.random() < 0.90:

        path.append("home_view")

        # Search
        if random.random() < 0.80:

            path.append("search")

            # Product View
            if random.random() < 0.75:

                path.append("product_view")

                # Cart
                if random.random() < 0.55:

                    path.append("add_to_cart")

                    # Checkout
                    if random.random() < 0.65:

                        path.append("checkout")

                        purchase_prob = 0.80

                        purchase_prob *= traffic_multiplier[source]
                        purchase_prob *= device_multiplier[device]

                        if variant == "New_UI":
                            purchase_prob *= 1.10

                        purchase_prob = min(
                            purchase_prob,
                            0.98
                        )

                        if random.random() < purchase_prob:

                            path.append("purchase")

                        else:

                            path.append(
                                "payment_failed"
                            )

    for event in path:

        current_time += timedelta(
            seconds=random.randint(
                5,
                120
            )
        )

        events.append([
            event_id,
            session_id,
            user_id,
            current_time,
            event,
            product_id
        ])

        # order creation

        if event == "purchase":

            product_price = products_df.loc[
                products_df["product_id"]
                == product_id,
                "price"
            ].iloc[0]

            quantity = random.randint(
                1,
                3
            )

            discount = random.choice(
                [0,5,10,15,20]
            )

            revenue = (
                product_price *
                quantity *
                (1 - discount/100)
            )

            orders.append([
                order_id,
                user_id,
                session_id,
                product_id,
                quantity,
                product_price,
                discount,
                round(revenue,2),
                current_time
            ])

            order_id += 1

        event_id += 1

events_df = pd.DataFrame(
    events,
    columns=[
        "event_id",
        "session_id",
        "user_id",
        "event_timestamp",
        "event_name",
        "product_id"
    ]
)

orders_df = pd.DataFrame(
    orders,
    columns=[
        "order_id",
        "user_id",
        "session_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "revenue",
        "order_timestamp"
    ]
)

print("\nFINAL COUNTS")
print("Users:", len(users_df))
print("Products:", len(products_df))
print("Sessions:", len(sessions_df))
print("Events:", len(events_df))
print("Orders:", len(orders_df))

users_df.to_csv(
    "users.csv",
    index=False
)

products_df.to_csv(
    "products.csv",
    index=False
)

sessions_df.to_csv(
    "sessions.csv",
    index=False
)

events_df.to_csv(
    "events.csv",
    index=False
)

orders_df.to_csv(
    "orders.csv",
    index=False
)

print("\nCSV Files Generated Successfully")


FINAL COUNTS
Users: 5000
Products: 500
Sessions: 48545
Events: 186064
Orders: 7980

CSV Files Generated Successfully


In [4]:
events_df["event_name"].value_counts()

event_name
app_open          48545
home_view         43649
search            34882
product_view      26106
add_to_cart       14250
checkout           9316
purchase           7980
payment_failed     1336
Name: count, dtype: int64

In [5]:
events_df.groupby("event_name")["user_id"].nunique()

event_name
add_to_cart       4258
app_open          5000
checkout          3678
home_view         4993
payment_failed     974
product_view      4815
purchase          3434
search            4939
Name: user_id, dtype: int64

In [6]:
buyers = events_df[
    events_df["event_name"]=="purchase"
]["user_id"].nunique()

total_users = users_df["user_id"].nunique()

print(
    "Buyer Rate:",
    round(
        buyers/total_users*100,
        2
    ),
    "%"
)

Buyer Rate: 68.68 %


In [7]:
orders_df["revenue"].describe()

count      7980.000000
mean      20392.343584
std       41845.434204
min          90.400000
25%        2033.025000
50%        4739.825000
75%       13421.600000
max      290673.000000
Name: revenue, dtype: float64

In [8]:
orders_df.shape

(7980, 9)

In [9]:
orders_df["order_bucket"] = pd.cut(
    orders_df["revenue"],
    bins=[0,1000,5000,10000,50000,1000000],
    labels=[
        "0-1K",
        "1K-5K",
        "5K-10K",
        "10K-50K",
        "50K+"
    ]
)

In [10]:
orders_df.describe

<bound method NDFrame.describe of       order_id  user_id  session_id  product_id  quantity  unit_price  \
0            1        1           3         398         3        2416   
1            2        3          14          91         3        7463   
2            3        4          16         391         3        6218   
3            4        4          21         172         2        1473   
4            5        5          37         150         2        1950   
...        ...      ...         ...         ...       ...         ...   
7975      7976     4997       48517         343         1        5637   
7976      7977     4997       48518         495         1       11372   
7977      7978     4999       48533         360         1         689   
7978      7979     4999       48536          45         1        2987   
7979      7980     5000       48542         376         2       44132   

      discount_pct   revenue     order_timestamp order_bucket  
0               20   5798

In [11]:
orders_df.to_csv(
    "orders1.csv",
    index=False
)